# **Factory-to-Customer Shipping Route Efficiency Analysis**

Nassau Candy Distributor

Business Analyst Internship — Unified Mentor

**Project Objective**

This project analyzes factory-to-customer shipping performance using order, shipment, geographic, product, and financial data. The analysis focuses on shipping lead time, route efficiency, geographic bottlenecks, and shipping mode performance to generate data-driven operational insights.

**Tools Used**

Python | Pandas | Streamlit | Data Visualization

**Analysis Workflow**

Data Loading → Data Validation → Data Cleaning → Feature Engineering → Route Analysis → KPI Analysis → Visualization → Streamlit Dashboard → Business Insights

# **1. Dataset Loading and Initial Overview**

The Nassau Candy Distributor dataset is loaded into a Pandas DataFrame. The initial row and column counts are checked to understand the size and structure of the dataset before beginning the analysis.

In [ ]:
# Import Pandas for data manipulation and analysis
import pandas as pd

# Load the raw Nassau Candy Distributor dataset
# into a Pandas DataFrame for analysis
df = pd.read_csv("Nassau Candy Distributor.csv")

# Display the basic dataset dimensions
# to understand the number of records and fields
print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 10194
Columns: 18


# **2. Initial Data Preview**

The first five records are reviewed to understand the available fields, data values, and overall structure of the dataset.

In [ ]:
# Display the first five records to get an initial
# understanding of the dataset structure and values
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Country/Region,City,State/Province,Postal Code,Division,Region,Product ID,Product Name,Sales,Units,Gross Profit,Cost
0,1,US-2021-103800-CHO-MIL-31000,03-01-2024,30-06-2026,Standard Class,103800,United States,Houston,Texas,77095,Chocolate,Interior,CHO-MIL-31000,Wonka Bar - Milk Chocolate,6.50,2,4.22,2.28
1,2,US-2021-112326-CHO-TRI-54000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,7.50,2,4.90,2.60
2,3,US-2021-112326-CHO-NUT-13000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-NUT-13000,Wonka Bar - Nutty Crunch Surprise,10.47,3,7.47,3.00
3,4,US-2021-112326-CHO-SCR-58000,04-01-2024,01-07-2026,Standard Class,112326,United States,Naperville,Illinois,60540,Chocolate,Interior,CHO-SCR-58000,Wonka Bar -Scrumdiddlyumptious,10.80,3,7.50,3.30
4,5,US-2021-141817-CHO-TRI-54000,05-01-2024,05-07-2026,Standard Class,141817,United States,Philadelphia,Pennsylvania,19143,Chocolate,Atlantic,CHO-TRI-54000,Wonka Bar - Triple Dazzle Caramel,11.25,3,7.35,3.90


# **3. Dataset Fields**

The column names are reviewed to identify the variables available for customer, shipment, geographic, product, sales, and cost analysis.

In [ ]:
# Display all column names available in the dataset
# to identify the variables that can be used for analysis
print(df.columns.tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']


# **4. Missing Value Check**

Missing values are checked across all fields to determine whether incomplete records require treatment before further analysis.

In [ ]:
# Check each column for missing values.
# This helps identify incomplete records that may
# require treatment during data cleaning.
missing_values = df.isnull().sum()

missing_values

,0
Row ID,0
Order ID,0
Order Date,0
Ship Date,0
Ship Mode,0
Customer ID,0
Country/Region,0
City,0
State/Province,0
Postal Code,0


# **5. Shipping Mode Distribution**

The distribution of orders across different shipping modes is examined. This provides an initial understanding of how frequently each shipping method is used.

In [ ]:
# Count the number of records for each shipping mode
# to understand the distribution of shipment methods
ship_mode_counts = df["Ship Mode"].value_counts()

ship_mode_counts

,count
Ship Mode,
Standard Class,6120
Second Class,1979
First Class,1548
Same Day,547


# **6. Regional Distribution**

The number of orders across customer regions is examined to understand the geographic distribution of the dataset.

In [ ]:
# Count the number of records in each customer region
# to understand the geographic distribution of orders
region_counts = df["Region"].value_counts()

region_counts

,count
Region,
Pacific,3253
Atlantic,2986
Interior,2335
Gulf,1620


# **7. Dataset Structure and Data Types**

The dataset structure and data types are reviewed before data cleaning. In particular, date fields are checked because they will be required to calculate shipping lead time.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10194 entries, 0 to 10193
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          10194 non-null  int64  
 1   Order ID        10194 non-null  object 
 2   Order Date      10194 non-null  object 
 3   Ship Date       10194 non-null  object 
 4   Ship Mode       10194 non-null  object 
 5   Customer ID     10194 non-null  int64  
 6   Country/Region  10194 non-null  object 
 7   City            10194 non-null  object 
 8   State/Province  10194 non-null  object 
 9   Postal Code     10194 non-null  object 
 10  Division        10194 non-null  object 
 11  Region          10194 non-null  object 
 12  Product ID      10194 non-null  object 
 13  Product Name    10194 non-null  object 
 14  Sales           10194 non-null  float64
 15  Units           10194 non-null  int64  
 16  Gross Profit    10194 non-null  float64
 17  Cost            10194 non-null 

# **8. Data Cleaning and Validation**

The dataset's date fields are converted from text into a standardized datetime format. This is necessary because the project requires calculating the number of days between order placement and shipment.

In [ ]:
# Convert Order Date and Ship Date from text format
# into datetime format so that date calculations
# can be performed accurately.
df["Order Date"] = pd.to_datetime(
    df["Order Date"],
    format="mixed",
    errors="coerce"
)

df["Ship Date"] = pd.to_datetime(
    df["Ship Date"],
    format="mixed",
    errors="coerce"
)

print("Dates converted!")
print()
print(df[["Order Date", "Ship Date"]].dtypes)

Dates converted!

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object


# **Date Validation**

The converted date fields are checked for missing values. This ensures that valid dates are available before calculating shipping lead time.

In [ ]:
print("Missing Order Dates:", df["Order Date"].isna().sum())
print("Missing Ship Dates:", df["Ship Date"].isna().sum())

Missing Order Dates: 0
Missing Ship Dates: 0


# **Shipping Lead Time Calculation**

Shipping Lead Time represents the number of days between the customer's order date and the shipment date. This KPI will be used throughout the project to evaluate route and shipping efficiency.

In [ ]:
# Calculate the number of days between the order date
# and the shipment date.
# Shipping Lead Time = Ship Date - Order Date
df["Shipping Lead Time"] = (
    df["Ship Date"] - df["Order Date"]
).dt.days

print("Shipping Lead Time calculated!")

Shipping Lead Time calculated!


# **Lead Time Validation**

A sample of records is reviewed to verify that the calculated shipping lead time is based on the difference between the order and shipment dates.

In [ ]:
df[["Order Date", "Ship Date", "Shipping Lead Time"]].head(10)

,Order Date,Ship Date,Shipping Lead Time
0,2024-03-01,2026-06-30,851
1,2024-04-01,2026-01-07,646
2,2024-04-01,2026-01-07,646
3,2024-04-01,2026-01-07,646
4,2024-05-01,2026-05-07,736
5,2024-06-01,2026-03-07,644
6,2024-06-01,2026-03-07,644
7,2024-06-01,2026-06-30,759
8,2024-06-01,2026-03-07,644
9,2024-06-01,2026-03-07,644


# **Invalid Lead Time Check**

Records with negative shipping lead times are checked because a negative value would indicate that an order was shipped before it was placed, which would be an invalid sequence.

In [ ]:
# Check for invalid records where the shipment date
# occurs before the order date.
negative_lead_times = (
    df["Shipping Lead Time"] < 0
).sum()

print("Negative Lead Times:", negative_lead_times)

Negative Lead Times: 0


# **Duplicate Record Check**

The dataset is checked for completely duplicated rows. Duplicate records could distort shipment counts and other calculated metrics.

In [ ]:
# Check for completely duplicated rows
# to ensure that identical records are not repeated.
duplicate_rows = df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


# **Cleaned Dataset**

After completing the initial validation and creating the Shipping Lead Time field, the processed dataset is saved as a separate CSV file. The original raw dataset is kept unchanged.

In [ ]:
df.to_csv("Nassau_Candy_Cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


# **Feature Engineering**

Feature engineering involves creating new analytical fields from the existing dataset. In this project, the main objective is to connect products with their corresponding factories and define factory-to-customer shipping routes. These fields will later be used to measure route efficiency and identify geographic bottlenecks.

In [ ]:
# Product-to-Factory Mapping
# This mapping is provided in the project instructions.
# It connects each product to the factory responsible for producing it.

product_factory_map = {
    "Wonka Bar - Nutty Crunch Surprise": "Lot's O' Nuts",
    "Wonka Bar - Fudge Mallows": "Lot's O' Nuts",
    "Wonka Bar - Scrumdiddlyumptious": "Lot's O' Nuts",
    "Wonka Bar - Milk Chocolate": "Wicked Choccy's",
    "Wonka Bar - Triple Dazzle Caramel": "Wicked Choccy's",
    "Laffy Taffy": "Sugar Shack",
    "SweetTARTS": "Sugar Shack",
    "Nerds": "Sugar Shack",
    "Fun Dip": "Sugar Shack",
    "Fizzy Lifting Drinks": "Sugar Shack",
    "Everlasting Gobstopper": "Secret Factory",
    "Hair Toffee": "The Other Factory",
    "Lickable Wallpaper": "Secret Factory",
    "Wonka Gum": "Secret Factory",
    "Kazookles": "The Other Factory"
}

In [ ]:
# Assign each product to its corresponding factory
# using the official Product-to-Factory mapping from the project instructions.

df["Factory"] = df["Product Name"].map(product_factory_map)

print("Factory column added successfully!")

Factory column added successfully!


In [ ]:
# Check whether any products could not be matched to a factory.
# A missing factory means the product name did not match our mapping.

print("Products without a factory:", df["Factory"].isna().sum())

Products without a factory: 2074


In [ ]:
# Identify the product names that were not matched to a factory.
# This helps us compare the dataset products with the official project mapping.

unmapped_products = df.loc[
    df["Factory"].isna(),
    "Product Name"
].unique()

print("Unmapped Products:")
print(unmapped_products)

Unmapped Products:
['Wonka Bar -Scrumdiddlyumptious' 'SweeTARTS']


In [ ]:
# Show the exact product names, including hidden spaces or characters.
# This helps us identify why these two products did not match.

for product in unmapped_products:
    print(repr(product))

'Wonka Bar -Scrumdiddlyumptious'
'SweeTARTS'


In [ ]:
# Check the exact product names stored in our Product-to-Factory mapping.

for product in product_factory_map:
    if "Scrumdiddly" in product or "Sweet" in product:
        print(repr(product), "->", product_factory_map[product])

'Wonka Bar - Scrumdiddlyumptious' -> Lot's O' Nuts
'SweetTARTS' -> Sugar Shack


In [ ]:
# Fix the two product names so they exactly match the dataset.
# The factory assignments remain the same as provided in the project instructions.

product_factory_map["Wonka Bar -Scrumdiddlyumptious"] = product_factory_map.pop(
    "Wonka Bar - Scrumdiddlyumptious"
)

product_factory_map["SweeTARTS"] = product_factory_map.pop(
    "SweetTARTS"
)

print("Product name mismatches fixed!")

Product name mismatches fixed!


In [ ]:
# Re-apply the corrected Product-to-Factory mapping.
# This updates the Factory column using the fixed product names.

df["Factory"] = df["Product Name"].map(product_factory_map)

print("Factory column updated successfully!")

Factory column updated successfully!


In [ ]:
# Final validation of the Product-to-Factory mapping.
# We expect every product in the dataset to have a factory.

print("Products without a factory:", df["Factory"].isna().sum())

Products without a factory: 0


In [ ]:
# Count the number of shipments handled by each factory.
# This gives us an initial view of the distribution of shipments across factories.

factory_distribution = df["Factory"].value_counts()

print("Shipments by Factory:")
print(factory_distribution)

Shipments by Factory:
Factory
Lot's O' Nuts        5692
Wicked Choccy's      4152
Secret Factory        217
The Other Factory     100
Sugar Shack            33
Name: count, dtype: int64


In [ ]:
# Create a route identifier using the factory and destination state.
# This will allow us to compare shipping performance across different routes.

df["Route"] = (
    df["Factory"] + " → " + df["State/Province"].astype(str)
)

print("Route column created successfully!")

Route column created successfully!


In [ ]:
# Check the number of unique shipping routes in the dataset.
# This tells us how many Factory → State combinations we have.

print("Number of unique routes:", df["Route"].nunique())

print("\nSample routes:")
print(df["Route"].unique()[:10])

Number of unique routes: 196

Sample routes:
["Wicked Choccy's → Texas" "Wicked Choccy's → Illinois"
 "Lot's O' Nuts → Illinois" "Wicked Choccy's → Pennsylvania"
 "Lot's O' Nuts → Kentucky" "Wicked Choccy's → Kentucky"
 "Wicked Choccy's → Georgia" "Wicked Choccy's → California"
 "Wicked Choccy's → Virginia" 'Secret Factory → Virginia']


In [ ]:
# Create a route-level performance summary.
# This helps us compare shipping efficiency across different Factory → State routes.

route_summary = df.groupby("Route").agg(
    Total_Shipments=("Route", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Variability=("Shipping Lead Time", "std")
).reset_index()

# Round the calculated values for easier reading.
route_summary["Average_Lead_Time"] = route_summary["Average_Lead_Time"].round(2)
route_summary["Lead_Time_Variability"] = route_summary["Lead_Time_Variability"].round(2)

print("Route performance summary created!")
print(route_summary.head(10))

Route performance summary created!
                                  Route  Total_Shipments  Average_Lead_Time  \
0               Lot's O' Nuts → Alabama               34            1272.76   
1               Lot's O' Nuts → Alberta               16            1309.50   
2               Lot's O' Nuts → Arizona              111            1310.34   
3              Lot's O' Nuts → Arkansas               31            1301.19   
4      Lot's O' Nuts → British Columbia               18            1346.22   
5            Lot's O' Nuts → California             1125            1340.93   
6              Lot's O' Nuts → Colorado              103            1395.16   
7           Lot's O' Nuts → Connecticut               47            1443.94   
8              Lot's O' Nuts → Delaware               43            1258.26   
9  Lot's O' Nuts → District of Columbia                8            1307.75   

   Lead_Time_Variability  
0                 276.22  
1                 272.00  
2             

In [ ]:
# Save the route-level performance summary for later analysis
# and for use in the Streamlit dashboard.

route_summary.to_csv(
    "Route_Performance_Summary.csv",
    index=False
)

print("Route performance summary saved successfully!")

Route performance summary saved successfully!


# **Overall Shipping Performance**

In [ ]:
# Calculate overall shipping lead-time statistics.
# These values give us a baseline for evaluating shipping performance.

overall_shipping_stats = df["Shipping Lead Time"].agg(
    ["mean", "median", "min", "max"]
)

print("Overall Shipping Lead-Time Statistics:")
print(overall_shipping_stats)

Overall Shipping Lead-Time Statistics:
mean      1344.239356
median    1276.000000
min        491.000000
max       1965.000000
Name: Shipping Lead Time, dtype: float64


# **Shipping performance by Ship Mode**

In [ ]:
# Compare shipping performance across different shipping modes.
# This helps us understand how lead time varies by delivery method.

ship_mode_analysis = df.groupby("Ship Mode").agg(
    Total_Shipments=("Ship Mode", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median")
).reset_index()

# Round lead-time values for easier reading.
ship_mode_analysis["Average_Lead_Time"] = (
    ship_mode_analysis["Average_Lead_Time"].round(2)
)

ship_mode_analysis["Median_Lead_Time"] = (
    ship_mode_analysis["Median_Lead_Time"].round(2)
)

print("Shipping Performance by Ship Mode:")
print(ship_mode_analysis)

Shipping Performance by Ship Mode:
        Ship Mode  Total_Shipments  Average_Lead_Time  Median_Lead_Time
0     First Class             1548            1360.13            1299.0
1        Same Day              547            1356.49            1297.0
2    Second Class             1979            1352.12            1300.0
3  Standard Class             6120            1336.58            1276.0


# **Shipping performance by Region**

In [ ]:
# Compare shipping performance across different regions.
# This helps identify regional differences in shipping volume and lead time.

region_analysis = df.groupby("Region").agg(
    Total_Shipments=("Region", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median")
).reset_index()

# Round lead-time values for easier reading.
region_analysis["Average_Lead_Time"] = (
    region_analysis["Average_Lead_Time"].round(2)
)

region_analysis["Median_Lead_Time"] = (
    region_analysis["Median_Lead_Time"].round(2)
)

print("Shipping Performance by Region:")
print(region_analysis)

Shipping Performance by Region:
     Region  Total_Shipments  Average_Lead_Time  Median_Lead_Time
0  Atlantic             2986            1350.57            1299.0
1      Gulf             1620            1323.10            1275.0
2  Interior             2335            1343.31            1276.0
3   Pacific             3253            1349.62            1284.0


# **Factory performance**

In [ ]:
# Compare shipping performance across factories.
# This helps us understand how shipment volume and lead time vary by factory.

factory_analysis = df.groupby("Factory").agg(
    Total_Shipments=("Factory", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median")
).reset_index()

# Round lead-time values for easier reading.
factory_analysis["Average_Lead_Time"] = (
    factory_analysis["Average_Lead_Time"].round(2)
)

factory_analysis["Median_Lead_Time"] = (
    factory_analysis["Median_Lead_Time"].round(2)
)

print("Shipping Performance by Factory:")
print(factory_analysis)

Shipping Performance by Factory:
             Factory  Total_Shipments  Average_Lead_Time  Median_Lead_Time
0      Lot's O' Nuts             5692            1344.91            1284.0
1     Secret Factory              217            1344.72            1301.0
2        Sugar Shack               33            1348.33            1275.0
3  The Other Factory              100            1298.60            1275.0
4    Wicked Choccy's             4152            1344.36            1276.0


# **Save our analysis tables**

In [ ]:
# Save the analysis tables created during Day 4.
# These files will be useful later for the dashboard and final report.

ship_mode_analysis.to_csv("Ship_Mode_Analysis.csv", index=False)
region_analysis.to_csv("Region_Analysis.csv", index=False)
factory_analysis.to_csv("Factory_Analysis.csv", index=False)

print("analysis tables saved successfully!")

analysis tables saved successfully!


# **Find the highest-volume routes**

In [ ]:
# Identify the routes with the highest shipment volume.
# High-volume routes are important because improvements on these routes
# could affect a larger number of shipments.

top_volume_routes = route_summary.sort_values(
    by="Total_Shipments",
    ascending=False
).head(10)

print("Top 10 Routes by Shipment Volume:")
print(top_volume_routes)

Top 10 Routes by Shipment Volume:
                            Route  Total_Shipments  Average_Lead_Time  \
5      Lot's O' Nuts → California             1125            1340.93   
144  Wicked Choccy's → California              823            1345.37   
34       Lot's O' Nuts → New York              645            1368.25   
49          Lot's O' Nuts → Texas              569            1322.77   
173    Wicked Choccy's → New York              436            1347.34   
189       Wicked Choccy's → Texas              388            1320.79   
42   Lot's O' Nuts → Pennsylvania              331            1341.68   
53     Lot's O' Nuts → Washington              269            1375.28   
13       Lot's O' Nuts → Illinois              267            1337.56   
38           Lot's O' Nuts → Ohio              262            1346.27   

     Lead_Time_Variability  
5                   302.96  
144                 297.52  
34                  275.32  
49                  286.59  
173              

In [ ]:
# Identify routes with the highest average shipping lead time.
# These routes have the longest average lead time in the dataset.

slowest_routes = route_summary.sort_values(
    by="Average_Lead_Time",
    ascending=False
).head(10)

print("Top 10 Routes by Average Lead Time:")
print(slowest_routes)

Top 10 Routes by Average Lead Time:
                               Route  Total_Shipments  Average_Lead_Time  \
69         Secret Factory → Kentucky                2             1888.0   
103         Sugar Shack → New Jersey                1             1820.0   
97          Sugar Shack → California                1             1815.0   
78    Secret Factory → New Hampshire                1             1700.0   
175   Wicked Choccy's → North Dakota                2             1666.0   
37      Lot's O' Nuts → North Dakota                5             1655.6   
98         Sugar Shack → Connecticut                1             1641.0   
194  Wicked Choccy's → West Virginia                2             1639.0   
110         Sugar Shack → Washington                1             1638.0   
63      Secret Factory → Connecticut                1             1638.0   

     Lead_Time_Variability  
69                   19.80  
103                    NaN  
97                     NaN  
78         

# **Focus on routes with enough shipments**

In [ ]:
# Filter routes with at least 10 shipments.
# This reduces the influence of routes with very small sample sizes.

reliable_routes = route_summary[
    route_summary["Total_Shipments"] >= 10
].copy()

print("Total routes:", len(route_summary))
print("Routes with at least 10 shipments:", len(reliable_routes))

Total routes: 196
Routes with at least 10 shipments: 96


# **Highest average lead time among reliable routes**

In [ ]:
# Identify the routes with the highest average lead time
# among routes that have at least 10 shipments.

reliable_slowest_routes = reliable_routes.sort_values(
    by="Average_Lead_Time",
    ascending=False
).head(10)

print("Top 10 Routes by Average Lead Time (10+ Shipments):")
print(reliable_slowest_routes)

Top 10 Routes by Average Lead Time (10+ Shipments):
                            Route  Total_Shipments  Average_Lead_Time  \
172  Wicked Choccy's → New Mexico               17            1516.71   
15           Lot's O' Nuts → Iowa               16            1516.06   
33     Lot's O' Nuts → New Mexico               18            1483.33   
26       Lot's O' Nuts → Missouri               36            1455.86   
153     Wicked Choccy's → Indiana               54            1447.67   
7     Lot's O' Nuts → Connecticut               47            1443.94   
159    Wicked Choccy's → Manitoba               10            1440.60   
48      Lot's O' Nuts → Tennessee              109            1439.91   
28       Lot's O' Nuts → Nebraska               23            1437.61   
94    Secret Factory → Washington               11            1431.91   

     Lead_Time_Variability  
172                 218.19  
15                  147.77  
33                  283.93  
26                  341.35  

# **Highest route variability**

In [ ]:
# Identify routes with the highest variation in shipping lead time.
# Higher variability means shipping times differ more from shipment to shipment.

highest_variability_routes = reliable_routes.sort_values(
    by="Lead_Time_Variability",
    ascending=False
).head(10)

print("Top 10 Routes by Lead-Time Variability (10+ Shipments):")
print(highest_variability_routes)

Top 10 Routes by Lead-Time Variability (10+ Shipments):
                               Route  Total_Shipments  Average_Lead_Time  \
67         Secret Factory → Illinois               13            1307.77   
61       Secret Factory → California               31            1327.19   
165       Wicked Choccy's → Missouri               29            1348.83   
12             Lot's O' Nuts → Idaho               14            1341.00   
17          Lot's O' Nuts → Kentucky               77            1319.32   
157      Wicked Choccy's → Louisiana               15            1278.07   
113   The Other Factory → California               21            1361.33   
156       Wicked Choccy's → Kentucky               58            1328.93   
179        Wicked Choccy's → Ontario               30            1213.37   
161  Wicked Choccy's → Massachusetts               54            1393.78   

     Lead_Time_Variability  
67                  404.95  
61                  380.85  
165                 

# **Find routes that have both high lead time AND high variability**

In [ ]:
# Identify routes that are above the median for both
# average lead time and lead-time variability.
# These routes deserve closer investigation because they show
# both longer shipping times and greater inconsistency.

lead_time_median = reliable_routes["Average_Lead_Time"].median()
variability_median = reliable_routes["Lead_Time_Variability"].median()

route_risk = reliable_routes[
    (reliable_routes["Average_Lead_Time"] > lead_time_median) &
    (reliable_routes["Lead_Time_Variability"] > variability_median)
].copy()

route_risk = route_risk.sort_values(
    by=["Average_Lead_Time", "Lead_Time_Variability"],
    ascending=False
)

print("Routes above the median for both lead time and variability:")
print(route_risk.head(15))

print("\nNumber of routes meeting both conditions:", len(route_risk))

Routes above the median for both lead time and variability:
                               Route  Total_Shipments  Average_Lead_Time  \
26          Lot's O' Nuts → Missouri               36            1455.86   
48         Lot's O' Nuts → Tennessee              109            1439.91   
81         Secret Factory → New York               24            1424.00   
154           Wicked Choccy's → Iowa               14            1418.57   
184   Wicked Choccy's → Rhode Island               21            1410.38   
193     Wicked Choccy's → Washington              221            1399.76   
6           Lot's O' Nuts → Colorado              103            1395.16   
161  Wicked Choccy's → Massachusetts               54            1393.78   
128     The Other Factory → New York               14            1387.93   
53        Lot's O' Nuts → Washington              269            1375.28   
145       Wicked Choccy's → Colorado               70            1368.10   
113   The Other Factory → Ca

# **Save the route-risk analysis**

In [ ]:
# Save the routes that are above the median for both
# average lead time and lead-time variability.

route_risk.to_csv(
    "Route_Risk_Analysis.csv",
    index=False
)

print("Route risk analysis saved successfully!")

Route risk analysis saved successfully!


In [ ]:
# Focus on routes with at least 50 shipments.
# This helps us examine routes that have a substantial number of observations.

high_volume_routes = route_summary[
    route_summary["Total_Shipments"] >= 50
].copy()

# Sort by average lead time to see the higher-lead-time routes first.
high_volume_routes = high_volume_routes.sort_values(
    by="Average_Lead_Time",
    ascending=False
)

print("High-Volume Routes (50+ Shipments):")
print(high_volume_routes)

High-Volume Routes (50+ Shipments):
                                Route  Total_Shipments  Average_Lead_Time  \
153         Wicked Choccy's → Indiana               54            1447.67   
48          Lot's O' Nuts → Tennessee              109            1439.91   
188       Wicked Choccy's → Tennessee               67            1426.42   
55          Lot's O' Nuts → Wisconsin               58            1421.71   
14            Lot's O' Nuts → Indiana               90            1419.74   
21           Lot's O' Nuts → Maryland               60            1419.08   
193      Wicked Choccy's → Washington              221            1399.76   
6            Lot's O' Nuts → Colorado              103            1395.16   
161   Wicked Choccy's → Massachusetts               54            1393.78   
174  Wicked Choccy's → North Carolina              101            1393.72   
171      Wicked Choccy's → New Jersey               59            1388.85   
53         Lot's O' Nuts → Washington   

In [ ]:
# Save the high-volume route analysis for use in the
# research paper and Streamlit dashboard.

high_volume_routes.to_csv(
    "High_Volume_Route_Analysis.csv",
    index=False
)

print("High-volume route analysis saved successfully!")

High-volume route analysis saved successfully!


# **State-level shipping performance**

In [ ]:
# Compare shipping performance across destination states.
# This helps identify differences in shipment volume and lead time
# at a more detailed geographic level.

state_analysis = df.groupby("State/Province").agg(
    Total_Shipments=("State/Province", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median")
).reset_index()

# Round lead-time values for easier reading.
state_analysis["Average_Lead_Time"] = (
    state_analysis["Average_Lead_Time"].round(2)
)

state_analysis["Median_Lead_Time"] = (
    state_analysis["Median_Lead_Time"].round(2)
)

print("Shipping Performance by State:")
print(state_analysis)

Shipping Performance by State:
               State/Province  Total_Shipments  Average_Lead_Time  \
0                     Alabama               61            1287.43   
1                     Alberta               26            1311.23   
2                     Arizona              224            1320.16   
3                    Arkansas               60            1320.38   
4            British Columbia               22            1353.64   
5                  California             2001            1343.00   
6                    Colorado              182            1385.59   
7                 Connecticut               82            1389.29   
8                    Delaware               96            1240.94   
9        District of Columbia               10            1301.20   
10                    Florida              383            1296.46   
11                    Georgia              184            1340.17   
12                      Idaho               21            1358.14   
13 

# **Highest-volume states**

In [ ]:
# Identify the states with the highest shipment volume.
# High-volume states are important because they represent a larger
# share of the dataset and provide more observations for analysis.

top_states = state_analysis.sort_values(
    by="Total_Shipments",
    ascending=False
).head(15)

print("Top 15 States by Shipment Volume:")
print(top_states)

Top 15 States by Shipment Volume:
    State/Province  Total_Shipments  Average_Lead_Time  Median_Lead_Time
5       California             2001            1343.00            1276.0
34        New York             1128            1361.20            1332.0
51           Texas              985            1319.82            1274.0
43    Pennsylvania              587            1348.00            1276.0
55      Washington              506            1386.29            1357.0
13        Illinois              492            1335.90            1276.0
39            Ohio              469            1350.74            1276.0
10         Florida              383            1296.46            1274.0
23        Michigan              255            1327.64            1275.0
36  North Carolina              249            1378.60            1358.0
2          Arizona              224            1320.16            1275.0
54        Virginia              224            1239.16            1273.0
11         Georgi

# **Compare high-volume states by lead time**

In [ ]:
# Focus on states with at least 100 shipments.
# This provides a stronger basis for comparing state-level
# shipping performance.

high_volume_states = state_analysis[
    state_analysis["Total_Shipments"] >= 100
].copy()

# Sort states by average lead time from highest to lowest.
high_volume_states = high_volume_states.sort_values(
    by="Average_Lead_Time",
    ascending=False
)

print("State Performance for States with 100+ Shipments:")
print(high_volume_states)

State Performance for States with 100+ Shipments:
    State/Province  Total_Shipments  Average_Lead_Time  Median_Lead_Time
50       Tennessee              183            1431.96            1390.0
14         Indiana              149            1431.87            1449.0
21        Maryland              105            1388.34            1363.0
55      Washington              506            1386.29            1357.0
6         Colorado              182            1385.59            1276.0
36  North Carolina              249            1378.60            1358.0
57       Wisconsin              110            1372.04            1302.0
32      New Jersey              130            1370.18            1275.0
34        New York             1128            1361.20            1332.0
39            Ohio              469            1350.74            1276.0
22   Massachusetts              135            1348.36            1274.0
43    Pennsylvania              587            1348.00            1276.0
4

# **Analyze Ship Mode by Region**

In [ ]:
# Analyze shipping performance across different regions and ship modes.
# This helps identify whether shipping patterns vary by geography
# and the selected shipping method.

region_shipmode_analysis = df.groupby(
    ["Region", "Ship Mode"]
).agg(
    Total_Shipments=("Ship Mode", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median")
).reset_index()

# Round lead-time values for easier interpretation.
region_shipmode_analysis["Average_Lead_Time"] = (
    region_shipmode_analysis["Average_Lead_Time"].round(2)
)

region_shipmode_analysis["Median_Lead_Time"] = (
    region_shipmode_analysis["Median_Lead_Time"].round(2)
)

print("Shipping Performance by Region and Ship Mode:")
print(region_shipmode_analysis)

Shipping Performance by Region and Ship Mode:
      Region       Ship Mode  Total_Shipments  Average_Lead_Time  \
0   Atlantic     First Class              496            1363.20   
1   Atlantic        Same Day              159            1322.97   
2   Atlantic    Second Class              548            1370.80   
3   Atlantic  Standard Class             1783            1343.30   
4       Gulf     First Class              234            1311.54   
5       Gulf        Same Day               83            1301.23   
6       Gulf    Second Class              329            1356.57   
7       Gulf  Standard Class              974            1316.44   
8   Interior     First Class              303            1371.00   
9   Interior        Same Day              120            1408.48   
10  Interior    Second Class              465            1327.37   
11  Interior  Standard Class             1447            1337.23   
12   Pacific     First Class              515            1372.86   
13

In [ ]:
# Save the Region × Ship Mode analysis for later use
# in the dashboard and research paper.

region_shipmode_analysis.to_csv(
    "Region_ShipMode_Analysis.csv",
    index=False
)

print("Region × Ship Mode analysis saved successfully!")

Region × Ship Mode analysis saved successfully!


# **Create a simple geographic summary**

In [ ]:
# Identify the highest and lowest observed average lead times
# among states with at least 100 shipments.
# Using a shipment threshold helps avoid conclusions from
# very small samples.

highest_avg_state = high_volume_states.iloc[0]

lowest_avg_state = high_volume_states.iloc[-1]

print("State with highest observed average lead time:")
print(
    highest_avg_state[
        ["State/Province", "Total_Shipments", "Average_Lead_Time"]
    ]
)

print()

print("State with lowest observed average lead time:")
print(
    lowest_avg_state[
        ["State/Province", "Total_Shipments", "Average_Lead_Time"]
    ]
)

State with highest observed average lead time:
State/Province       Tennessee
Total_Shipments            183
Average_Lead_Time      1431.96
Name: 50, dtype: object

State with lowest observed average lead time:
State/Province       Virginia
Total_Shipments           224
Average_Lead_Time     1239.16
Name: 54, dtype: object


# **Product-level shipping performance**

In [ ]:
# Analyze shipping performance at the product level.
# This helps us understand how different products
# contribute to shipment volume and shipping lead time.

product_analysis = df.groupby("Product Name").agg(
    Total_Shipments=("Product Name", "count"),
    Total_Units=("Units", "sum"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median"),
    Total_Sales=("Sales", "sum"),
    Total_Gross_Profit=("Gross Profit", "sum")
).reset_index()

# Round numerical values for easier interpretation.
product_analysis["Average_Lead_Time"] = (
    product_analysis["Average_Lead_Time"].round(2)
)

product_analysis["Median_Lead_Time"] = (
    product_analysis["Median_Lead_Time"].round(2)
)

product_analysis["Total_Sales"] = (
    product_analysis["Total_Sales"].round(2)
)

product_analysis["Total_Gross_Profit"] = (
    product_analysis["Total_Gross_Profit"].round(2)
)

print("Product-Level Shipping Performance:")
print(product_analysis)

Product-Level Shipping Performance:
                         Product Name  Total_Shipments  Total_Units  \
0              Everlasting Gobstopper                3           13   
1                Fizzy Lifting Drinks                6           21   
2                             Fun Dip                3            8   
3                         Hair Toffee                4           17   
4                           Kazookles               96          371   
5                         Laffy Taffy               10           27   
6                  Lickable Wallpaper               94          393   
7                               Nerds                4           10   
8                           SweeTARTS               10           41   
9           Wonka Bar - Fudge Mallows             1818         6914   
10         Wonka Bar - Milk Chocolate             2137         8267   
11  Wonka Bar - Nutty Crunch Surprise             1810         6755   
12  Wonka Bar - Triple Dazzle Caramel    

# **Factory-Level Performance**

In [ ]:
# Analyze overall business and shipping performance by factory.
# This helps connect factory activity with shipment volume,
# shipping lead time, sales, and profitability.

factory_business_analysis = df.groupby("Factory").agg(
    Total_Shipments=("Factory", "count"),
    Total_Units=("Units", "sum"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Median_Lead_Time=("Shipping Lead Time", "median"),
    Total_Sales=("Sales", "sum"),
    Total_Gross_Profit=("Gross Profit", "sum")
).reset_index()

# Round numerical values for easier interpretation.
factory_business_analysis["Average_Lead_Time"] = (
    factory_business_analysis["Average_Lead_Time"].round(2)
)

factory_business_analysis["Median_Lead_Time"] = (
    factory_business_analysis["Median_Lead_Time"].round(2)
)

factory_business_analysis["Total_Sales"] = (
    factory_business_analysis["Total_Sales"].round(2)
)

factory_business_analysis["Total_Gross_Profit"] = (
    factory_business_analysis["Total_Gross_Profit"].round(2)
)

print("Factory-Level Business and Shipping Performance:")
print(factory_business_analysis)

Factory-Level Business and Shipping Performance:
             Factory  Total_Shipments  Total_Units  Average_Lead_Time  \
0      Lot's O' Nuts             5692        21412            1344.91   
1     Secret Factory              217          884            1344.72   
2        Sugar Shack               33          107            1348.33   
3  The Other Factory              100          388            1298.60   
4    Wicked Choccy's             4152        15863            1344.36   

   Median_Lead_Time  Total_Sales  Total_Gross_Profit  
0            1284.0     76340.15            52771.05  
1            1301.0      8587.50             4344.70  
2            1275.0       220.98              121.23  
3            1275.0      1282.25              152.25  
4            1276.0     55352.75            36053.57  


In [ ]:
# Analyze product performance within each factory.
# This helps identify which products contribute most
# to shipment volume and business performance at each factory.

product_factory_analysis = df.groupby(
    ["Factory", "Product Name"]
).agg(
    Total_Shipments=("Product Name", "count"),
    Total_Units=("Units", "sum"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Total_Sales=("Sales", "sum"),
    Total_Gross_Profit=("Gross Profit", "sum")
).reset_index()

# Round numerical values for easier interpretation.
product_factory_analysis["Average_Lead_Time"] = (
    product_factory_analysis["Average_Lead_Time"].round(2)
)

product_factory_analysis["Total_Sales"] = (
    product_factory_analysis["Total_Sales"].round(2)
)

product_factory_analysis["Total_Gross_Profit"] = (
    product_factory_analysis["Total_Gross_Profit"].round(2)
)

print("Product-to-Factory Performance:")
print(product_factory_analysis)

Product-to-Factory Performance:
              Factory                       Product Name  Total_Shipments  \
0       Lot's O' Nuts          Wonka Bar - Fudge Mallows             1818   
1       Lot's O' Nuts  Wonka Bar - Nutty Crunch Surprise             1810   
2       Lot's O' Nuts     Wonka Bar -Scrumdiddlyumptious             2064   
3      Secret Factory             Everlasting Gobstopper                3   
4      Secret Factory                 Lickable Wallpaper               94   
5      Secret Factory                          Wonka Gum              120   
6         Sugar Shack               Fizzy Lifting Drinks                6   
7         Sugar Shack                            Fun Dip                3   
8         Sugar Shack                        Laffy Taffy               10   
9         Sugar Shack                              Nerds                4   
10        Sugar Shack                          SweeTARTS               10   
11  The Other Factory                       

In [ ]:
# Save the product-to-factory analysis for later use
# in the dashboard and research paper.

product_factory_analysis.to_csv(
    "Product_Factory_Analysis.csv",
    index=False
)

print("Product-to-Factory analysis saved successfully!")

Product-to-Factory analysis saved successfully!


# **Identify high-volume products**

In [ ]:
# Focus on products with at least 100 shipments.
# This provides a stronger basis for comparing product-level
# shipping performance.

high_volume_products = product_analysis[
    product_analysis["Total_Shipments"] >= 100
].copy()

# Sort products by shipment volume from highest to lowest.
high_volume_products = high_volume_products.sort_values(
    by="Total_Shipments",
    ascending=False
)

print("Products with 100+ Shipments:")
print(high_volume_products)

Products with 100+ Shipments:
                         Product Name  Total_Shipments  Total_Units  \
10         Wonka Bar - Milk Chocolate             2137         8267   
13     Wonka Bar -Scrumdiddlyumptious             2064         7743   
12  Wonka Bar - Triple Dazzle Caramel             2015         7596   
9           Wonka Bar - Fudge Mallows             1818         6914   
11  Wonka Bar - Nutty Crunch Surprise             1810         6755   
14                          Wonka Gum              120          478   

    Average_Lead_Time  Median_Lead_Time  Total_Sales  Total_Gross_Profit  
10            1339.90            1276.0     26867.75            17443.37  
13            1343.43            1276.0     27874.80            19357.50  
12            1349.09            1284.0     28485.00            18610.20  
9             1339.35            1276.0     24890.40            16593.60  
11            1352.18            1302.0     23574.95            16819.95  
14            1308.59 

In [ ]:
# Save the high-volume product analysis for later use
# in the dashboard, research paper, and executive summary.

high_volume_products.to_csv(
    "High_Volume_Product_Analysis.csv",
    index=False
)

print("High-volume product analysis saved successfully!")

High-volume product analysis saved successfully!


# **Factory Contribution %**

In [ ]:
# Calculate each factory's contribution to overall shipments,
# sales, and gross profit.
# This helps connect shipping activity with business impact.

factory_contribution = factory_business_analysis.copy()

# Calculate percentage contribution for each business metric.
factory_contribution["Shipment_Share_%"] = (
    factory_contribution["Total_Shipments"]
    / factory_contribution["Total_Shipments"].sum()
    * 100
)

factory_contribution["Sales_Share_%"] = (
    factory_contribution["Total_Sales"]
    / factory_contribution["Total_Sales"].sum()
    * 100
)

factory_contribution["Profit_Share_%"] = (
    factory_contribution["Total_Gross_Profit"]
    / factory_contribution["Total_Gross_Profit"].sum()
    * 100
)

# Round percentages for easier interpretation.
factory_contribution["Shipment_Share_%"] = (
    factory_contribution["Shipment_Share_%"].round(2)
)

factory_contribution["Sales_Share_%"] = (
    factory_contribution["Sales_Share_%"].round(2)
)

factory_contribution["Profit_Share_%"] = (
    factory_contribution["Profit_Share_%"].round(2)
)

print("Factory Contribution Analysis:")
print(
    factory_contribution[
        [
            "Factory",
            "Total_Shipments",
            "Shipment_Share_%",
            "Total_Sales",
            "Sales_Share_%",
            "Total_Gross_Profit",
            "Profit_Share_%"
        ]
    ]
)

Factory Contribution Analysis:
             Factory  Total_Shipments  Shipment_Share_%  Total_Sales  \
0      Lot's O' Nuts             5692             55.84     76340.15   
1     Secret Factory              217              2.13      8587.50   
2        Sugar Shack               33              0.32       220.98   
3  The Other Factory              100              0.98      1282.25   
4    Wicked Choccy's             4152             40.73     55352.75   

   Sales_Share_%  Total_Gross_Profit  Profit_Share_%  
0          53.84            52771.05           56.47  
1           6.06             4344.70            4.65  
2           0.16              121.23            0.13  
3           0.90              152.25            0.16  
4          39.04            36053.57           38.58  


In [ ]:
# Save the factory contribution analysis for use
# in the dashboard and final research paper.

factory_contribution.to_csv(
    "Factory_Contribution_Analysis.csv",
    index=False
)

print("Factory contribution analysis saved successfully!")

Factory contribution analysis saved successfully!


In [ ]:
# Create a folder for the Streamlit dashboard.
!mkdir -p dashboard

print("Dashboard folder created successfully!")

Dashboard folder created successfully!


In [ ]:
# Check the project folders currently available in Colab.
!ls

 dashboard			     Product_Factory_Analysis.csv
 Factory_Analysis.csv		     Region_Analysis.csv
 Factory_Contribution_Analysis.csv   Region_ShipMode_Analysis.csv
 High_Volume_Product_Analysis.csv    Route_Performance_Summary.csv
 High_Volume_Route_Analysis.csv      Route_Risk_Analysis.csv
 Nassau_Candy_Cleaned.csv	     sample_data
'Nassau Candy Distributor.csv'	     Ship_Mode_Analysis.csv


In [ ]:
%%writefile dashboard/app.py

# Import Streamlit for building the interactive dashboard.
import streamlit as st

# Configure the dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)

# Main dashboard title.
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

# Introduction to the dashboard.
st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)

# Confirmation message.
st.success("Dashboard setup completed successfully!")

Writing dashboard/app.py


In [ ]:
# Confirm that the Streamlit application file was created.
!ls dashboard

app.py


In [ ]:
# Create a folder to store the dataset used by the dashboard.
!mkdir -p data

print("Data folder created successfully!")

Data folder created successfully!


In [ ]:
# Copy the cleaned dataset into the dashboard project's data folder.
!cp Nassau_Candy_Cleaned.csv data/Nassau_Candy_Cleaned.csv

print("Cleaned dataset copied successfully!")

Cleaned dataset copied successfully!


In [ ]:
# Confirm that the cleaned dataset is available in the data folder.
!ls data

Nassau_Candy_Cleaned.csv


In [ ]:
%%writefile dashboard/app.py

# Import required libraries.
import pandas as pd
import streamlit as st

# Configure the Streamlit dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)

# Load the cleaned dataset
# Read the cleaned dataset from the project's data folder.
df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")


# Dashboard Header
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)

# Display a small confirmation that the dataset loaded.
st.success("Dataset loaded successfully!")

# Display the number of records currently loaded.
st.write(f"Total records: **{len(df):,}**")

Overwriting dashboard/app.py


In [ ]:
# Check that the updated Streamlit application was written correctly.

!head -30 dashboard/app.py


# Import required libraries.
import pandas as pd
import streamlit as st

# Configure the Streamlit dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)

# Load the cleaned dataset
# Read the cleaned dataset from the project's data folder.
df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")


# Dashboard Header
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)

# Display a small confirmation that the dataset loaded.
st.success("Dataset loaded successfully!")


# **Install Streamlit**

In [ ]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 69.1 MB/s eta 0:00:00


# **Run Streamlit**

In [ ]:
# Start the Streamlit dashboard.
!streamlit run dashboard/app.py &>/content/streamlit.log &

In [ ]:
# Check whether the Streamlit server started successfully.
!cat /content/streamlit.log

In [ ]:
from google.colab import output

# Open the Streamlit application through Colab's
# built-in port forwarding.
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
!streamlit run dashboard/app.py &>/content/streamlit.log &

# **Check the Streamlit error log**

In [ ]:
# Check the latest Streamlit server messages/errors.
!tail -50 /content/streamlit.log

# **Stop the current Streamlit server**

In [ ]:
# Stop the currently running Streamlit server.
!pkill -f "streamlit run" || true

print("Old Streamlit server stopped.")

^C
Old Streamlit server stopped.


# **Start Streamlit with Colab-compatible settings**

In [ ]:
# Start Streamlit with settings that allow Colab's
# forwarded browser connection to communicate with the app.

!streamlit run dashboard/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

# **Check the Server**

In [ ]:
# Check that Streamlit started successfully.
!tail -20 /content/streamlit.log

# **Open it through Colab**

In [ ]:
from google.colab import output

# Open Streamlit through Colab's secure port forwarding.
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [ ]:
%%writefile dashboard/app.py

# Import required libraries.
import pandas as pd
import streamlit as st

# Configure the Streamlit dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)


# Load the cleaned dataset
# Read the cleaned dataset from the project's data folder.
df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")


# Dashboard Header
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)


# Key Performance Indicators
# Calculate the main dashboard metrics.
total_shipments = len(df)

average_lead_time = df["Shipping Lead Time"].mean()

total_factories = df["Factory"].nunique()

total_regions = df["Region"].nunique()

# Display the KPIs in four columns.
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "📦 Total Shipments",
        f"{total_shipments:,}"
    )

with col2:
    st.metric(
        "🚚 Avg. Lead Time",
        f"{average_lead_time:,.2f} days"
    )

with col3:
    st.metric(
        "🏭 Factories",
        total_factories
    )

with col4:
    st.metric(
        "📍 Regions",
        total_regions
    )


# Dataset Confirmation
st.success("Dataset loaded successfully!")

Overwriting dashboard/app.py


In [ ]:
# Save the current dataframe again.
# This version includes the Factory and Route columns
# created during the later analysis steps.

df.to_csv(
    "data/Nassau_Candy_Cleaned.csv",
    index=False
)

print("Updated dashboard dataset saved successfully!")
print("Columns:", len(df.columns))

Updated dashboard dataset saved successfully!
Columns: 21


In [ ]:
# Confirm that the dashboard dataset contains
# the columns required by the Streamlit application.

print("Factory column exists:", "Factory" in df.columns)
print("Route column exists:", "Route" in df.columns)

print()
print("Factory values:")
print(df["Factory"].value_counts())

Factory column exists: True
Route column exists: True

Factory values:
Factory
Lot's O' Nuts        5692
Wicked Choccy's      4152
Secret Factory        217
The Other Factory     100
Sugar Shack            33
Name: count, dtype: int64


In [ ]:
# Reload the dashboard dataset directly from the saved CSV
# to confirm that the file itself contains the Factory column.

dashboard_df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")

print("Factory exists in saved CSV:",
      "Factory" in dashboard_df.columns)

print("Route exists in saved CSV:",
      "Route" in dashboard_df.columns)

print("Number of records:",
      len(dashboard_df))

Factory exists in saved CSV: True
Route exists in saved CSV: True
Number of records: 10194


In [ ]:
%%writefile dashboard/app.py

# Import required libraries.
import pandas as pd
import streamlit as st

# Configure the Streamlit dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)


# Load Dataset
# Read the cleaned dataset used by the dashboard.
df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")


# Dashboard Header
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)


# Sidebar Filters
st.sidebar.header("🔎 Dashboard Filters")

# Factory filter.
factory_options = ["All"] + sorted(df["Factory"].dropna().unique().tolist())

selected_factory = st.sidebar.selectbox(
    "Select Factory",
    factory_options
)

# Region filter.
region_options = ["All"] + sorted(df["Region"].dropna().unique().tolist())

selected_region = st.sidebar.selectbox(
    "Select Region",
    region_options
)

# Ship Mode filter.
ship_mode_options = ["All"] + sorted(df["Ship Mode"].dropna().unique().tolist())

selected_ship_mode = st.sidebar.selectbox(
    "Select Ship Mode",
    ship_mode_options
)


# Apply Filters

filtered_df = df.copy()

if selected_factory != "All":
    filtered_df = filtered_df[
        filtered_df["Factory"] == selected_factory
    ]

if selected_region != "All":
    filtered_df = filtered_df[
        filtered_df["Region"] == selected_region
    ]

if selected_ship_mode != "All":
    filtered_df = filtered_df[
        filtered_df["Ship Mode"] == selected_ship_mode
    ]


# Key Performance Indicators
total_shipments = len(filtered_df)

average_lead_time = filtered_df["Shipping Lead Time"].mean()

total_factories = filtered_df["Factory"].nunique()

total_regions = filtered_df["Region"].nunique()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "📦 Total Shipments",
        f"{total_shipments:,}"
    )

with col2:
    st.metric(
        "🚚 Avg. Lead Time",
        f"{average_lead_time:,.2f} days"
    )

with col3:
    st.metric(
        "🏭 Factories",
        total_factories
    )

with col4:
    st.metric(
        "📍 Regions",
        total_regions
    )


# Filter Status
st.success(
    f"Showing {len(filtered_df):,} shipments based on the selected filters."
)

Overwriting dashboard/app.py


# **Shipments by Region 📊**

In [ ]:
%%writefile dashboard/app.py

# Import required libraries.
import pandas as pd
import streamlit as st

# Configure the Streamlit dashboard page.
st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍫",
    layout="wide"
)

# Load Dataset
# Read the cleaned dataset used by the dashboard.
df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")

# Dashboard Header
st.title("🍫 Nassau Candy Shipping Route Efficiency Analysis")

st.markdown(
    """
    This interactive dashboard analyzes factory-to-customer
    shipping performance, route efficiency, lead times,
    product activity, and factory contribution.
    """
)

# Sidebar Filters
st.sidebar.header("🔎 Dashboard Filters")

factory_options = ["All"] + sorted(
    df["Factory"].dropna().unique().tolist()
)

selected_factory = st.sidebar.selectbox(
    "Select Factory",
    factory_options
)

region_options = ["All"] + sorted(
    df["Region"].dropna().unique().tolist()
)

selected_region = st.sidebar.selectbox(
    "Select Region",
    region_options
)

ship_mode_options = ["All"] + sorted(
    df["Ship Mode"].dropna().unique().tolist()
)

selected_ship_mode = st.sidebar.selectbox(
    "Select Ship Mode",
    ship_mode_options
)


# Apply Filters
filtered_df = df.copy()

if selected_factory != "All":
    filtered_df = filtered_df[
        filtered_df["Factory"] == selected_factory
    ]

if selected_region != "All":
    filtered_df = filtered_df[
        filtered_df["Region"] == selected_region
    ]

if selected_ship_mode != "All":
    filtered_df = filtered_df[
        filtered_df["Ship Mode"] == selected_ship_mode
    ]

# Key Performance Indicators
total_shipments = len(filtered_df)

average_lead_time = filtered_df["Shipping Lead Time"].mean()

total_factories = filtered_df["Factory"].nunique()

total_regions = filtered_df["Region"].nunique()

col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric(
        "📦 Total Shipments",
        f"{total_shipments:,}"
    )

with col2:
    st.metric(
        "🚚 Avg. Lead Time",
        f"{average_lead_time:,.2f} days"
    )

with col3:
    st.metric(
        "🏭 Factories",
        total_factories
    )

with col4:
    st.metric(
        "📍 Regions",
        total_regions
    )

# Filter Status
st.success(
    f"Showing {len(filtered_df):,} shipments based on the selected filters."
)


# Shipments by Region
st.subheader("📊 Shipments by Region")

# Count shipments for each region based on the active filters.
region_shipments = (
    filtered_df["Region"]
    .value_counts()
    .rename_axis("Region")
    .reset_index(name="Shipments")
)

# Display the regional shipment distribution as a bar chart.
st.bar_chart(
    region_shipments.set_index("Region")
)

# Average Lead Time by Region
st.subheader("🚚 Average Lead Time by Region")

# Calculate the average shipping lead time for each region
# based on the active dashboard filters.
region_lead_time = (
    filtered_df.groupby("Region")["Shipping Lead Time"]
    .mean()
    .round(2)
    .reset_index()
)

# Display the average lead time by region.
st.bar_chart(
    region_lead_time.set_index("Region")
)

# Shipments by Ship Mode
st.subheader("🚚 Shipments by Ship Mode")

# Count shipments for each shipping mode based on
# the active dashboard filters.
ship_mode_shipments = (
    filtered_df["Ship Mode"]
    .value_counts()
    .rename_axis("Ship Mode")
    .reset_index(name="Shipments")
)

# Display shipment volume by shipping mode.
st.bar_chart(
    ship_mode_shipments.set_index("Ship Mode")
)

# Average Lead Time by Factory
st.subheader("🏭 Average Lead Time by Factory")

# Calculate the average shipping lead time for each factory
# based on the active dashboard filters.
factory_lead_time = (
    filtered_df.groupby("Factory")["Shipping Lead Time"]
    .mean()
    .round(2)
    .reset_index()
)

# Display average lead time by factory.
st.bar_chart(
    factory_lead_time.set_index("Factory")
)

# Shipments by Product
st.subheader("🍫 Shipments by Product")

# Count shipments for each product based on the active filters.
product_shipments = (
    filtered_df["Product Name"]
    .value_counts()
    .sort_values(ascending=True)
    .rename_axis("Product Name")
    .reset_index(name="Shipments")
)

# Display product shipment volume.
st.bar_chart(
    product_shipments.set_index("Product Name")
)

# Average Lead Time by Product
st.subheader("🚚 Average Lead Time by Product")

# Calculate average shipping lead time for each product.
product_lead_time = (
    filtered_df.groupby("Product Name")["Shipping Lead Time"]
    .mean()
    .round(2)
    .sort_values(ascending=True)
    .reset_index()
)

# Display average lead time by product.
st.bar_chart(
    product_lead_time.set_index("Product Name")
)

# Route Performance Analysis
st.subheader("🛣️ Route Performance")

# Create route-level shipping performance using the
# currently selected dashboard filters.
route_analysis = filtered_df.groupby("Route").agg(
    Total_Shipments=("Route", "count"),
    Average_Lead_Time=("Shipping Lead Time", "mean"),
    Lead_Time_Variability=("Shipping Lead Time", "std")
).reset_index()

# Keep routes with at least 10 shipments.
# This avoids overinterpreting very small samples.
reliable_routes = route_analysis[
    route_analysis["Total_Shipments"] >= 10
].copy()

# Round values for easier display.
reliable_routes["Average_Lead_Time"] = (
    reliable_routes["Average_Lead_Time"].round(2)
)

reliable_routes["Lead_Time_Variability"] = (
    reliable_routes["Lead_Time_Variability"].round(2)
)

# Sort routes by average lead time.
reliable_routes = reliable_routes.sort_values(
    by="Average_Lead_Time",
    ascending=False
)

# Display the 10 routes with the highest observed
# average lead time among routes with 10+ shipments.
st.dataframe(
    reliable_routes.head(10),
    use_container_width=True
)

# Factory Shipment Contribution
st.subheader("🏭 Factory Shipment Contribution")

# Count shipments by factory.
factory_shipments = (
    filtered_df["Factory"]
    .value_counts()
    .rename_axis("Factory")
    .reset_index(name="Shipments")
)

# Calculate each factory's percentage of filtered shipments.
factory_shipments["Shipment_Share_%"] = (
    factory_shipments["Shipments"]
    / factory_shipments["Shipments"].sum()
    * 100
)

# Round percentages.
factory_shipments["Shipment_Share_%"] = (
    factory_shipments["Shipment_Share_%"].round(2)
)

# Display factory shipment contribution.
st.dataframe(
    factory_shipments,
    use_container_width=True
)

# Filtered Data Preview
st.subheader("📋 Filtered Shipment Data")

# Display the first 100 filtered records.
st.dataframe(
    filtered_df.head(100),
    use_container_width=True
)

# Download Filtered Data
csv_data = filtered_df.to_csv(index=False).encode("utf-8")

st.download_button(
    label="⬇️ Download Filtered Data",
    data=csv_data,
    file_name="Nassau_Candy_Filtered_Data.csv",
    mime="text/csv"
)

# Dashboard Footer
st.markdown("---")

st.caption(
    "Nassau Candy Distributor | Factory-to-Customer "
    "Shipping Route Efficiency Analysis"
)

Overwriting dashboard/app.py


In [ ]:
import os

print("dashboard folder:", os.path.exists("dashboard"))
print("app.py:", os.path.exists("dashboard/app.py"))
print("data folder:", os.path.exists("data"))
print("cleaned CSV:", os.path.exists("data/Nassau_Candy_Cleaned.csv"))

dashboard folder: True
app.py: True
data folder: True
cleaned CSV: True


In [84]:
from google.colab import files
import os
import shutil

# Open the upload window
uploaded = files.upload()

# Create the data folder
os.makedirs("data", exist_ok=True)

# Find the uploaded CSV
uploaded_file = list(uploaded.keys())[0]

# Put it where the dashboard expects it
shutil.copy(
    uploaded_file,
    "data/Nassau_Candy_Cleaned.csv"
)

print("✅ Dataset restored!")
print("File:", uploaded_file)
print(
    "Dashboard file exists:",
    os.path.exists("data/Nassau_Candy_Cleaned.csv")
)


KeyboardInterrupt



In [85]:
import pandas as pd

# Load the dataset currently used by the dashboard
dashboard_df = pd.read_csv("data/Nassau_Candy_Cleaned.csv")

print("Rows:", len(dashboard_df))
print("Columns:", len(dashboard_df.columns))
print()
print("Factory exists:", "Factory" in dashboard_df.columns)
print("Route exists:", "Route" in dashboard_df.columns)
print("Shipping Lead Time exists:", "Shipping Lead Time" in dashboard_df.columns)

Rows: 10194
Columns: 21

Factory exists: True
Route exists: True
Shipping Lead Time exists: True


In [86]:
!pkill -f "streamlit run" || true

!streamlit run dashboard/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

^C


In [87]:
from google.colab import output

output.serve_kernel_port_as_iframe(8501)


<IPython.core.display.Javascript object>

In [91]:
%%writefile dashboard/app.py

# ============================================================
# NASSAU CANDY DISTRIBUTOR
# Factory-to-Customer Shipping Route Efficiency Analysis
# ============================================================

import streamlit as st
import pandas as pd


# ------------------------------------------------------------
# PAGE CONFIGURATION
# ------------------------------------------------------------

st.set_page_config(
    page_title="Nassau Candy Shipping Analysis",
    page_icon="🍬",
    layout="wide"
)


# ------------------------------------------------------------
# DASHBOARD TITLE
# ------------------------------------------------------------

st.title("🍬 Nassau Candy Distributor")
st.subheader("Factory-to-Customer Shipping Route Efficiency Analysis")

st.markdown(
    """
    This dashboard analyzes shipping performance across factories,
    regions, ship modes, products, and factory-to-customer routes.
    """
)

st.divider()


# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

@st.cache_data
def load_data():

    df = pd.read_csv(
        "data/Nassau_Candy_Cleaned.csv"
    )

    # Convert date columns back to datetime after reading CSV
    df["Order Date"] = pd.to_datetime(
        df["Order Date"],
        errors="coerce"
    )

    df["Ship Date"] = pd.to_datetime(
        df["Ship Date"],
        errors="coerce"
    )

    return df


df = load_data()


# ------------------------------------------------------------
# SIDEBAR FILTERS
# ------------------------------------------------------------

st.sidebar.header("🔎 Dashboard Filters")


# Factory Filter
factory_options = ["All"] + sorted(
    df["Factory"].dropna().unique().tolist()
)

selected_factory = st.sidebar.selectbox(
    "Select Factory",
    factory_options
)


# Region Filter
region_options = ["All"] + sorted(
    df["Region"].dropna().unique().tolist()
)

selected_region = st.sidebar.selectbox(
    "Select Region",
    region_options
)


# Ship Mode Filter
ship_mode_options = ["All"] + sorted(
    df["Ship Mode"].dropna().unique().tolist()
)

selected_ship_mode = st.sidebar.selectbox(
    "Select Ship Mode",
    ship_mode_options
)


# Route Filter
route_options = ["All"] + sorted(
    df["Route"].dropna().unique().tolist()
)

selected_route = st.sidebar.selectbox(
    "Select Route",
    route_options
)


# ------------------------------------------------------------
# APPLY FILTERS
# ------------------------------------------------------------

filtered_df = df.copy()


if selected_factory != "All":

    filtered_df = filtered_df[
        filtered_df["Factory"] == selected_factory
    ]


if selected_region != "All":

    filtered_df = filtered_df[
        filtered_df["Region"] == selected_region
    ]


if selected_ship_mode != "All":

    filtered_df = filtered_df[
        filtered_df["Ship Mode"] == selected_ship_mode
    ]


if selected_route != "All":

    filtered_df = filtered_df[
        filtered_df["Route"] == selected_route
    ]


# ------------------------------------------------------------
# CHECK IF FILTERS RETURN DATA
# ------------------------------------------------------------

if filtered_df.empty:

    st.warning(
        "No shipment records are available for the selected filters."
    )

    st.stop()


# ------------------------------------------------------------
# KPI CARDS
# ------------------------------------------------------------

st.header("📊 Shipping Overview")


total_shipments = len(filtered_df)

average_lead_time = (
    filtered_df["Shipping Lead Time"].mean()
)

total_factories = (
    filtered_df["Factory"].nunique()
)

total_regions = (
    filtered_df["Region"].nunique()
)


col1, col2, col3, col4 = st.columns(4)


with col1:

    st.metric(
        "Total Shipments",
        f"{total_shipments:,}"
    )


with col2:

    st.metric(
        "Avg Lead Time",
        f"{average_lead_time:,.2f} Days"
    )


with col3:

    st.metric(
        "Factories",
        total_factories
    )


with col4:

    st.metric(
        "Regions",
        total_regions
    )


st.divider()


# ------------------------------------------------------------
# REGION ANALYSIS
# ------------------------------------------------------------

st.header("🌎 Regional Shipping Analysis")


region_summary = (
    filtered_df
    .groupby("Region")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Avg_Lead_Time=("Shipping Lead Time", "mean")
    )
    .reset_index()
)


col1, col2 = st.columns(2)


with col1:

    st.subheader("Shipments by Region")

    region_volume = (
        region_summary
        .set_index("Region")["Total_Shipments"]
    )

    st.bar_chart(
        region_volume
    )


with col2:

    st.subheader("Average Lead Time by Region")

    region_lead = (
        region_summary
        .set_index("Region")["Avg_Lead_Time"]
    )

    st.bar_chart(
        region_lead
    )


st.divider()


# ------------------------------------------------------------
# SHIP MODE ANALYSIS
# ------------------------------------------------------------

st.header("🚚 Ship Mode Analysis")


ship_mode_summary = (
    filtered_df
    .groupby("Ship Mode")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Avg_Lead_Time=("Shipping Lead Time", "mean")
    )
    .reset_index()
)


col1, col2 = st.columns(2)


with col1:

    st.subheader("Shipments by Ship Mode")

    ship_volume = (
        ship_mode_summary
        .set_index("Ship Mode")["Total_Shipments"]
    )

    st.bar_chart(
        ship_volume
    )


with col2:

    st.subheader("Average Lead Time by Ship Mode")

    ship_lead = (
        ship_mode_summary
        .set_index("Ship Mode")["Avg_Lead_Time"]
    )

    st.bar_chart(
        ship_lead
    )


st.divider()


# ------------------------------------------------------------
# FACTORY ANALYSIS
# ------------------------------------------------------------

st.header("🏭 Factory Performance")


factory_summary = (
    filtered_df
    .groupby("Factory")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Avg_Lead_Time=("Shipping Lead Time", "mean")
    )
    .reset_index()
)


col1, col2 = st.columns(2)


with col1:

    st.subheader("Shipments by Factory")

    factory_volume = (
        factory_summary
        .set_index("Factory")["Total_Shipments"]
    )

    st.bar_chart(
        factory_volume
    )


with col2:

    st.subheader("Average Lead Time by Factory")

    factory_lead = (
        factory_summary
        .set_index("Factory")["Avg_Lead_Time"]
    )

    st.bar_chart(
        factory_lead
    )


st.divider()


# ------------------------------------------------------------
# PRODUCT ANALYSIS
# ------------------------------------------------------------

st.header("🍫 Product Shipping Analysis")


product_summary = (
    filtered_df
    .groupby("Product Name")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Avg_Lead_Time=("Shipping Lead Time", "mean")
    )
    .reset_index()
)


col1, col2 = st.columns(2)


with col1:

    st.subheader("Top Products by Shipment Volume")

    top_products = (
        product_summary
        .sort_values(
            "Total_Shipments",
            ascending=False
        )
        .head(10)
        .set_index("Product Name")
    )

    st.bar_chart(
        top_products["Total_Shipments"]
    )


with col2:

    st.subheader("Average Lead Time by Product")

    product_lead = (
        product_summary
        .sort_values(
            "Avg_Lead_Time",
            ascending=False
        )
        .head(10)
        .set_index("Product Name")
    )

    st.bar_chart(
        product_lead["Avg_Lead_Time"]
    )


st.divider()


# ------------------------------------------------------------
# ROUTE PERFORMANCE VISUALIZATION
# DAY 9 ADDITION
# ------------------------------------------------------------

st.header("🛣️ Route Efficiency Analysis")


# Create route-level summary from currently filtered data
route_analysis = (
    filtered_df
    .groupby("Route")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Avg_Lead_Time=("Shipping Lead Time", "mean"),
        Median_Lead_Time=("Shipping Lead Time", "median"),
        Lead_Time_Variability=("Shipping Lead Time", "std")
    )
    .reset_index()
)


# Replace NaN variability for routes with only one shipment
route_analysis["Lead_Time_Variability"] = (
    route_analysis["Lead_Time_Variability"]
    .fillna(0)
)


# ------------------------------------------------------------
# ROUTE CHARTS
# ------------------------------------------------------------

col1, col2 = st.columns(2)


with col1:

    st.subheader(
        "Top 10 Routes by Shipment Volume"
    )

    top_volume_routes = (
        route_analysis
        .sort_values(
            "Total_Shipments",
            ascending=False
        )
        .head(10)
        .set_index("Route")
    )

    st.bar_chart(
        top_volume_routes["Total_Shipments"]
    )


with col2:

    st.subheader(
        "Top 10 Routes by Average Lead Time"
    )

    top_lead_routes = (
        route_analysis
        .sort_values(
            "Avg_Lead_Time",
            ascending=False
        )
        .head(10)
        .set_index("Route")
    )

    st.bar_chart(
        top_lead_routes["Avg_Lead_Time"]
    )


# ------------------------------------------------------------
# ROUTE PERFORMANCE TABLE
# ------------------------------------------------------------

st.subheader("Route Performance Details")


route_display = (
    route_analysis
    .sort_values(
        "Total_Shipments",
        ascending=False
    )
    .copy()
)


route_display["Avg_Lead_Time"] = (
    route_display["Avg_Lead_Time"].round(2)
)

route_display["Median_Lead_Time"] = (
    route_display["Median_Lead_Time"].round(2)
)

route_display["Lead_Time_Variability"] = (
    route_display["Lead_Time_Variability"].round(2)
)


st.dataframe(
    route_display,
    use_container_width=True,
    hide_index=True
)


st.divider()


# ------------------------------------------------------------
# FACTORY SHIPMENT CONTRIBUTION
# ------------------------------------------------------------

st.header("📦 Factory Shipment Contribution")


factory_contribution = (
    filtered_df
    .groupby("Factory")
    .agg(
        Total_Shipments=("Order ID", "count"),
        Total_Units=("Units", "sum"),
        Total_Sales=("Sales", "sum"),
        Gross_Profit=("Gross Profit", "sum")
    )
    .reset_index()
)


factory_contribution["Shipment_Contribution_%"] = (
    factory_contribution["Total_Shipments"]
    / factory_contribution["Total_Shipments"].sum()
    * 100
)


factory_contribution["Shipment_Contribution_%"] = (
    factory_contribution[
        "Shipment_Contribution_%"
    ].round(2)
)


st.dataframe(
    factory_contribution,
    use_container_width=True,
    hide_index=True
)


st.divider()


# ------------------------------------------------------------
# FILTERED SHIPMENT DATA
# ------------------------------------------------------------

st.header("📋 Filtered Shipment Data")


st.caption(
    f"Displaying {len(filtered_df):,} shipment records."
)


st.dataframe(
    filtered_df,
    use_container_width=True,
    hide_index=True
)


# ------------------------------------------------------------
# DOWNLOAD FILTERED DATA
# ------------------------------------------------------------

csv = filtered_df.to_csv(
    index=False
).encode("utf-8")


st.download_button(
    label="⬇️ Download Filtered Data",
    data=csv,
    file_name="Nassau_Candy_Filtered_Data.csv",
    mime="text/csv"
)


st.divider()


# ------------------------------------------------------------
# FOOTER
# ------------------------------------------------------------

st.caption(
    "Nassau Candy Distributor | "
    "Factory-to-Customer Shipping Route Efficiency Analysis"
)

Overwriting dashboard/app.py


In [92]:
!pkill -f "streamlit run" || true

!streamlit run dashboard/app.py \
    --server.port 8501 \
    --server.address 0.0.0.0 \
    --server.enableCORS false \
    --server.enableXsrfProtection false \
    > /content/streamlit.log 2>&1 &

^C


In [93]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [94]:
%%writefile requirements.txt

pandas
numpy
streamlit
plotly

Writing requirements.txt


In [95]:
%%writefile README.md

# 🍬 Nassau Candy Distributor – Shipping Route Efficiency Analysis

## 📌 Project Overview

This project analyzes the factory-to-customer shipping operations of Nassau Candy Distributor.

The objective is to understand shipping lead times, route performance, regional patterns, factory contribution, ship modes, and product-level shipping activity.

The analysis combines Python-based data analysis with an interactive Streamlit dashboard.

---

## 🎯 Project Objectives

- Analyze overall shipping performance
- Calculate shipping lead time
- Compare shipping performance across factories
- Analyze regional shipping patterns
- Compare different ship modes
- Identify high-volume shipping routes
- Identify routes with higher average lead times
- Analyze product-level shipment activity
- Understand factory contribution to shipments, sales, and profit
- Provide an interactive dashboard for business users

---

## 📊 Key Analysis Areas

### 1. Shipping Lead Time
Shipping lead time is calculated as:

`Ship Date - Order Date`

### 2. Factory Analysis
The project compares shipping activity and lead time across the assigned factories.

### 3. Regional Analysis
Shipping performance is analyzed across different regions and states/provinces.

### 4. Ship Mode Analysis
The dashboard compares:
- Standard Class
- Second Class
- First Class
- Same Day

### 5. Route Analysis
Factory-to-customer routes are created using:

`Factory → State/Province`

Routes are analyzed based on:
- Shipment volume
- Average lead time
- Median lead time
- Lead-time variability

### 6. Product Analysis
Products are analyzed based on shipment volume and average shipping lead time.

---

## 🛠️ Tools & Technologies

- Python
- Pandas
- NumPy
- Streamlit
- Matplotlib
- Jupyter / Google Colab
- CSV Dataset

---

## 📁 Project Structure

```text
Nassau_Candy_Shipping_Analysis
│
├── data
│   └── Nassau_Candy_Cleaned.csv
│
├── analysis
│   ├── Ship_Mode_Analysis.csv
│   ├── Region_Analysis.csv
│   ├── Factory_Analysis.csv
│   ├── Route_Performance_Summary.csv
│   ├── Route_Risk_Analysis.csv
│   ├── High_Volume_Route_Analysis.csv
│   ├── Product_Factory_Analysis.csv
│   └── Factory_Contribution_Analysis.csv
│
├── dashboard
│   └── app.py
│
├── report
│
├── screenshots
│
├── requirements.txt
├── README.md
└── Nassau_Candy_Shipping_Analysis.ipynb

Writing README.md


In [96]:
import os

os.makedirs("report", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

print("report/ folder created ✅")
print("screenshots/ folder created ✅")

report/ folder created ✅
screenshots/ folder created ✅


In [97]:
import os

print("📁 PROJECT FILES")
print("=" * 50)

for root, dirs, files in os.walk("."):
    # Ignore system folders
    if root.startswith("./."):
        continue

    level = root.count(os.sep)

    indent = "    " * level

    folder_name = os.path.basename(root) or "."

    print(f"{indent}📂 {folder_name}/")

    for file in files:
        if not file.startswith("."):
            print(f"{indent}    📄 {file}")

📁 PROJECT FILES
📂 ./
    📄 Route_Performance_Summary.csv
    📄 Region_Analysis.csv
    📄 Region_ShipMode_Analysis.csv
    📄 Route_Risk_Analysis.csv
    📄 Nassau Candy Distributor.csv
    📄 Ship_Mode_Analysis.csv
    📄 Factory_Contribution_Analysis.csv
    📄 streamlit.log
    📄 High_Volume_Route_Analysis.csv
    📄 Nassau_Candy_Cleaned.csv
    📄 Factory_Analysis.csv
    📄 README.md
    📄 requirements.txt
    📄 Product_Factory_Analysis.csv
    📄 High_Volume_Product_Analysis.csv
    📂 report/
    📂 dashboard/
        📄 app.py
    📂 screenshots/
    📂 data/
        📄 Nassau_Candy_Cleaned.csv
    📂 sample_data/
        📄 anscombe.json
        📄 README.md
        📄 mnist_train_small.csv
        📄 california_housing_train.csv
        📄 california_housing_test.csv
        📄 mnist_test.csv


In [98]:
%%writefile .gitignore

# Google Colab / Jupyter
.ipynb_checkpoints/
sample_data/

# Streamlit
streamlit.log
.streamlit/

# Python
__pycache__/
*.py[cod]
*.pyo

# OS files
.DS_Store
Thumbs.db

Writing .gitignore


In [99]:
import os

notebooks = [
    f for f in os.listdir(".")
    if f.endswith(".ipynb")
]

if notebooks:
    print("📓 Notebook found:")
    for notebook in notebooks:
        print("✅", notebook)
else:
    print("❌ No .ipynb file found in the project folder.")

❌ No .ipynb file found in the project folder.
